# 04 — Model Evaluation

Loads the saved model and runs the full evaluation suite:
- Classification metrics (AUC, F1, precision, recall, accuracy)
- Confusion matrix
- ROC curve
- SHAP global feature importance
- SHAP local waterfall for a single match

Run `03_training.ipynb` first to generate the saved model.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import pandas as pd

from conf.settings import PROCESSED_DIR, DATASET_FILE, TEST_SIZE, VAL_SIZE, RANDOM_STATE
from src.module.model.model import CSWinnerModel
from src.module.model.evaluate import ModelEvaluator
from src.module.model.explain import ModelExplainer
from src.module.model.features import build_features, get_feature_matrix
from utils.data_processing import split_data

%matplotlib inline

## 1. Load Saved Model

In [ ]:
model = CSWinnerModel.load()
print('Model loaded.')
print(f'Pipeline steps: {list(model.pipeline.named_steps.keys())}')

## 2. Reconstruct Test Split

Use the same random state and split sizes as training to get the identical test set.

In [ ]:
df = pd.read_parquet(PROCESSED_DIR / DATASET_FILE)
df = build_features(df)
X, y = get_feature_matrix(df)

_, _, X_test, _, _, y_test = split_data(
    X, y, test_size=TEST_SIZE, val_size=VAL_SIZE, random_state=RANDOM_STATE
)
print(f'Test set: {X_test.shape[0]:,} rows x {X_test.shape[1]} features')

## 3. Classification Metrics

In [ ]:
evaluator = ModelEvaluator(model)
metrics = evaluator.compute_metrics(X_test, y_test)

print('Test set metrics:')
for k, v in metrics.items():
    print(f'  {k:12s}: {v:.4f}')

## 4. Confusion Matrix

In [ ]:
evaluator.plot_confusion_matrix(X_test, y_test, filename='confusion_matrix.png')
plt.show()

## 5. ROC Curve

In [ ]:
evaluator.plot_roc_curve(X_test, y_test, filename='roc_curve.png')
plt.show()

## 6. SHAP — Global Feature Importance

Beeswarm plot: each point is one match, x-axis is the SHAP value (impact on prediction).
Red = high feature value, blue = low. Features sorted by mean absolute SHAP.

In [ ]:
explainer = ModelExplainer(model=model)
shap_values = explainer.shap_summary(df, save=True, filename='shap_summary.png')

## 7. SHAP — Local Explanation for One Match

Waterfall plot for the match where the model is most confident about team1 winning.

In [ ]:
probas = model.pipeline.predict_proba(X_test)[:, 1]
most_confident_idx = probas.argmax()

sample_row = df.iloc[X_test.index[most_confident_idx]]
result = explainer.explain_match(sample_row.to_dict(), save=True, filename='shap_waterfall.png')

print(f"Predicted winner: {result['predicted_winner']}")
print(f"P(team1 wins): {result['prob_team1_wins']:.1%}")
print(f"P(team2 wins): {result['prob_team2_wins']:.1%}")
print('\nTop reasons (SHAP):')
for r in result['top_reasons']:
    print(f"  {r['feature']:35s} SHAP={r['shap_value']:+.4f}  val={r['feature_value']:.4f}")

## 8. Probability Distribution on Test Set

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(8, 4))
for label, color, name in [(1, '#2ecc71', 'team1 wins'), (0, '#e74c3c', 'team2 wins')]:
    mask = y_test == label
    ax.hist(probas[mask], bins=50, alpha=0.6, color=color, label=name, density=True)
ax.axvline(0.5, color='black', linestyle='--', label='threshold')
ax.set_xlabel('P(team1 wins)')
ax.set_ylabel('Density')
ax.set_title('Predicted probability distribution by true outcome')
ax.legend()
plt.tight_layout()